In [16]:
from rank_bm25 import BM25Okapi
import json 
import os
import re
import pickle


In [18]:
# feed chunks_jsonl 
chunks_folder_path = "../../data/sample_chunks_jsonl"
bm25_file_path = "../../data/sample_bm25_index.pkl"
mapping_file_path = "../../data/sample_mapping_table.pkl"

In [ ]:
# demo 
corpus = [
    ["introduction", "to", "vector", "databases", "and", "ai"],
    ["how", "to", "configure", "and", "set","up", "qdrant", "locally"],
    ["resolving", "out", "of", "setup", "error", "qdrant", "python", "loops"]
]
bm25 = BM25Okapi(corpus)
# tokenise query :

query = "qdrant setup".split()
doc_scores = bm25.get_scores(query)
print(doc_scores)
best_match = bm25.get_top_n(query, corpus, n=2)
print(best_match)


[0.         0.49074951 0.        ]
[['how', 'to', 'configure', 'and', 'set', 'up', 'qdrant', 'locally'], ['resolving', 'out', 'of', 'setu', 'error', 'qdran', 'python', 'loops']]


In [ ]:
# clean chuncks text

def tokenise_chunks_text(text):
    clean_text = re.sub(r'[^\w\s]', ' ', text.lower())
    return clean_text.split()
    

In [15]:

corpus = []
mapping_table = []
for filename in os.listdir(chunks_folder_path):
    if filename.endswith(".jsonl"):
        file_path = os.path.join(chunks_folder_path, filename)
        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                data_item = json.loads(line)
                chunk_id = data_item['chunk_id']
                chunk_text = data_item['text']
                tokenise_text = tokenise_chunks_text(chunk_text)
                corpus.append(tokenise_text)
                mapping_table.append({
                    "chunk_id":chunk_id,
                    "text" : chunk_text
                })
                
                
                

print(f"Building master BM25 index for {len(corpus)} documents...")    
bm25 = BM25Okapi(corpus)               
print("All files processed and loaded successfully!")


Building master BM25 index for 473 documents...
All files processed and loaded successfully!


In [24]:
# sample: save 
with open(bm25_file_path, "wb") as f_bm25:
    pickle.dump(bm25, f_bm25)

with open(mapping_file_path, "wb") as f_map:
    pickle.dump(mapping_table, f_map)

